# Chapter 7

Add your content here.

In [1]:

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

# 1. Prepare Data
def get_data(n_samples=2000):
    # Target: 2 Moons
    X1, _ = make_moons(n_samples=n_samples, noise=0.05)
    X1 = torch.tensor(X1, dtype=torch.float32) * 2 - 1 # Scale nicely
    
    # Source: Gaussian Noise
    X0 = torch.randn_like(X1)
    return X0, X1

# 2. Define the Velocity Model (Simple MLP)
class VelocityField(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64), # Input: x, y, time
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 2)  # Output: velocity_x, velocity_y
        )
    
    def forward(self, x, t):
        # We concatenate time to the input coordinates
        t_embed = t.view(-1, 1).expand(x.shape[0], 1)
        inp = torch.cat([x, t_embed], dim=1)
        return self.net(inp)

# 3. ODE Solver (Euler Method)
# This simulates the flow from t=0 to t=1
def ode_solve(model, z0, steps=20):
    z = z0.clone()
    dt = 1.0 / steps
    traj = [z.clone()]
    for i in range(steps):
        t = torch.tensor(i * dt).float()
        with torch.no_grad():
            v = model(z, t)
        z = z + v * dt # Update position: pos + velocity * time
        traj.append(z.clone())
    return z, traj

# 4. Training Loop
def train_rectified_flow(model, pairs_X0, pairs_X1, epochs=500):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    batch_size = 256
    
    for epoch in range(epochs):
        # Random time t between 0 and 1
        t = torch.rand(batch_size)
        
        # Select random batch indices
        idx = torch.randint(0, len(pairs_X0), (batch_size,))
        x0_batch = pairs_X0[idx]
        x1_batch = pairs_X1[idx]
        
        # Linear Interpolation (The straight line assumption)
        # z_t = t * x1 + (1-t) * x0
        # We need to reshape t for broadcasting
        t_expanded = t.view(-1, 1)
        z_t = t_expanded * x1_batch + (1 - t_expanded) * x0_batch
        
        # Calculate Target Velocity
        # The straight line direction is simply (Target - Source)
        target_v = x1_batch - x0_batch
        
        # Predict Velocity
        pred_v = model(z_t, t)
        
        # Loss: Mean Squared Error
        loss = torch.mean((pred_v - target_v)**2)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# --- Main Execution ---

# Step A: 1-Rectified Flow
print("Training 1-Rectified Flow...")
X0, X1 = get_data()
model_1 = VelocityField()
train_rectified_flow(model_1, X0, X1, epochs=1000)

# Step B: Reflow (Generate new pairs using Model 1)
print("Generating Reflow Pairs...")
# We move X0 through Model 1 to get where the model *thinks* X1 is.
Z1_reflow, _ = ode_solve(model_1, X0, steps=100)

# Step C: 2-Rectified Flow (Train on (X0, Z1_reflow))
print("Training 2-Rectified Flow...")
model_2 = VelocityField()
train_rectified_flow(model_2, X0, Z1_reflow, epochs=1000)

print("Done! You have performed Rectification and Distillation.")

Training 1-Rectified Flow...
Epoch 0, Loss: 3.5226
Epoch 100, Loss: 2.2591
Epoch 200, Loss: 2.2490
Epoch 300, Loss: 2.2700
Epoch 400, Loss: 1.8944
Epoch 500, Loss: 2.1637
Epoch 600, Loss: 2.1187
Epoch 700, Loss: 1.9885
Epoch 800, Loss: 1.8341
Epoch 900, Loss: 2.0489
Generating Reflow Pairs...
Training 2-Rectified Flow...
Epoch 0, Loss: 0.4560
Epoch 100, Loss: 0.0676
Epoch 200, Loss: 0.0364
Epoch 300, Loss: 0.0199
Epoch 400, Loss: 0.0141
Epoch 500, Loss: 0.0091
Epoch 600, Loss: 0.0075
Epoch 700, Loss: 0.0056
Epoch 800, Loss: 0.0046
Epoch 900, Loss: 0.0040
Done! You have performed Rectification and Distillation.
